# 15 — Tokenization: characters, words and BPE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/015-tokenization.ipynb)

A model never sees text, only the symbols a tokenizer slices it into. In this notebook you start from single characters on a four-word corpus, learn merge rules the way BPE does, and watch the vocabulary grow one symbol at a time. Then you see what word-level tokenizing costs when a word it never met walks in.

## 1. The numbers: a corpus you can hold

Four short words are the whole training set. Splitting them into characters gives the smallest possible vocabulary, and the list of characters is the alphabet the tokenizer is allowed to know. Count the characters and you have the size of the starting vocabulary.

In [ ]:
corpus = "low lower lowest"                      # the entire training text
words = corpus.split()                         # split on spaces -> three words
chars = sorted(set(corpus.replace(" ", "")))   # every distinct character, sorted
print("words:", words)                         # the three words
print("character vocabulary:", chars)          # 7 symbols, one per letter used
print("vocabulary size:", len(chars))          # 7: that is where BPE starts
print("as symbols:", [list(w) for w in words])  # each word chopped into characters
print("counts:", {w: words.count(w) for w in words})  # each word appears once

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>corpus = ...</code> — the whole training text, low and lower and lowest.</li><li><code>words = corpus.split()</code> — split on whitespace; three words come out.</li><li><code>chars = sorted(set(...))</code> — <code>set</code> removes duplicates, <code>sorted</code> fixes the order.</li><li><code>list(w)</code> — turns the string <code>w</code> into a list of one-character symbols.</li><li><code>{w: words.count(w) ...}</code> — a dictionary from each word to how often it appears.</li></ul>

</details>

## 2. Step by step: merge the most frequent pair

BPE looks at every adjacent pair of symbols and counts how often it occurs across the corpus. The winner is glued into one new symbol, every word is rewritten, and the loop starts again. Each round adds exactly one symbol to the vocabulary.

In [ ]:
from collections import Counter                   # counts things in one line

vocab = {tuple(w): words.count(w) for w in words}   # word as a symbol tuple -> frequency
sizes = [len(chars)]                                # vocabulary size before any merge

for step in range(8):                               # eight rounds, one merge each
    pairs = Counter()                               # fresh tally for this round
    for w, n in vocab.items():                      # every distinct word and its count
        for a, b in zip(w, w[1:]):                  # every adjacent pair inside the word
            pairs[(a, b)] += n                      # weight the pair by the word's frequency
    if not pairs:                                   # nothing left to merge
        break                                       # stop early
    best, count = pairs.most_common(1)[0]           # the single most frequent pair
    merged = best[0] + best[1]                      # glue the two symbols into one
    new_vocab = {}                                  # corpus after the merge
    for w, n in vocab.items():                      # rewrite word by word
        out = []                                    # symbols of the rewritten word
        i = 0                                       # cursor inside the word
        while i < len(w):                           # scan left to right
            if i < len(w) - 1 and (w[i], w[i + 1]) == best:   # is this the winning pair?
                out.append(merged)                  # replace two symbols with one
                i += 2                              # skip past both halves
            else:                                   # any other position
                out.append(w[i])                    # copy the symbol unchanged
                i += 1                              # advance one step
        new_vocab[tuple(out)] = n                   # store the rewritten word
    vocab = new_vocab                               # swap in the new corpus
    sizes.append(sizes[-1] + 1)                     # one new symbol -> vocabulary grows by one
    print(f"merge {step + 1}: {best} -> {merged} (count {count})")   # report the round

print("final words:", [" ".join(w) for w in vocab])   # words as symbol sequences
print("vocabulary size:", sizes[-1])                 # 7 characters plus the merges that fit

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>Counter</code> — a dictionary that counts, perfect for pair tallies.</li><li><code>vocab = {...}</code> — the corpus as tuples of symbols, with frequencies.</li><li><code>sizes = [len(chars)]</code> — vocabulary size at merge count zero.</li><li><code>pairs[(a, b)] += n</code> — a pair seen in a frequent word counts more.</li><li><code>pairs.most_common(1)[0]</code> — the winner and its count.</li><li><code>merged = best[0] + best[1]</code> — the new symbol is a concatenation.</li><li><code>out.append(merged); i += 2</code> — replace the pair, jump over it.</li><li><code>sizes.append(sizes[-1] + 1)</code> — every merge adds exactly one symbol.</li></ul>

</details>

## 3. Plot: vocabulary size against merges

Each merge rule buys exactly one new symbol, so the curve is a straight line. The slope is the trade the tokenizer makes: fewer symbols per word, one more entry in the table. A real tokenizer just keeps turning this crank for tens of thousands of rounds.

In [ ]:
import numpy as np                              # the array library, preinstalled
import matplotlib.pyplot as plt                 # plotting library, preinstalled

xs = np.arange(len(sizes))                      # one x per merge count, 0 to 8
plt.figure(figsize=(6, 3))                      # a wide, short figure
plt.plot(xs, sizes, marker="o", color="tab:blue")   # vocabulary size after each merge
plt.xlabel("number of merges")                  # x-axis: rules learned
plt.ylabel("symbols in the vocabulary")         # y-axis: distinct symbols
plt.title("Each merge buys one symbol: 7 -> 13")  # the takeaway
plt.grid(alpha=0.3)                             # faint grid, easier to read
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>np.arange(len(sizes))</code> — x values 0, 1, 2, ... one per merge count.</li><li><code>plt.plot(...)</code> — the vocabulary size curve, dots at each merge.</li><li><code>plt.xlabel</code> / <code>plt.ylabel</code> — name both axes with units.</li><li><code>plt.title</code> — states the takeaway, not the axes.</li></ul>

</details>

## 4. Interactive: choose how many merges

Drag the slider to stop the merge loop early or let it run to the end. This corpus only offers six pairs worth merging, so the last two slider positions change nothing. Each word is shown as its symbols after that many merges, so words grow chunky and the shared `low` appears in all three.

In [ ]:
from ipywidgets import interact                   # sliders that rerun a function

def tokens_after(n_merges=4):                   # n_merges comes from the slider
    v = {tuple(w): 1 for w in words}            # every word starts as characters
    for _ in range(n_merges):                   # learn this many rules
        counts = Counter()                      # pair tally for one round
        for w in v:                             # every word in the corpus
            for a, b in zip(w, w[1:]):          # adjacent symbols only
                counts[(a, b)] += 1             # one vote per occurrence
        if not counts:                          # nothing left to merge
            break                               # stop early
        pair = counts.most_common(1)[0][0]      # this round's winner
        nv = {}                                 # rewritten corpus
        for w in v:                             # rewrite each word
            out = []                            # output symbols
            i = 0                               # cursor into the word
            while i < len(w):                   # scan left to right
                if i < len(w) - 1 and (w[i], w[i + 1]) == pair:   # the winning pair?
                    out.append(w[i] + w[i + 1])  # glue into one symbol
                    i += 2                      # skip both halves
                else:                           # otherwise
                    out.append(w[i])            # copy the symbol
                    i += 1                      # advance one
            nv[tuple(out)] = 1                  # store the rewritten word
        v = nv                                  # move to the new corpus
    n_sym = len(set(s for w in v for s in w))   # distinct symbols actually in use
    txt = "\n".join(" ".join(w) for w in v)     # one line per word, symbols spaced out
    plt.figure(figsize=(6, 2.4))                # a short strip, not a chart
    plt.axis("off")                             # no axes: this is a text panel
    plt.title(f"{n_merges} merges -> vocabulary of {n_sym} symbols")   # live readout
    plt.text(0.01, 0.5, txt, family="monospace", fontsize=13, va="center")   # the words
    plt.show()                                  # draw the frame

interact(tokens_after, n_merges=(0, 8, 1))      # slider: 0 to 8 merges

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>interact</code> — attaches a slider to a function that redraws.</li><li><code>def tokens_after(n_merges=4)</code> — one frame per slider value.</li><li><code>v = {tuple(w): 1 ...}</code> — the corpus, every word as characters.</li><li><code>counts[(a, b)] += 1</code> — tally adjacent pairs.</li><li><code>counts.most_common(1)[0][0]</code> — the pair to merge this round.</li><li><code>out.append(w[i] + w[i + 1])</code> — glue the pair into a single symbol.</li><li><code>n_sym = len(set(...))</code> — count the distinct symbols left in the corpus.</li><li><code>plt.axis("off")</code> — hide axes; the picture is text.</li><li><code>plt.title(f"...{n_sym}...")</code> — the readout changes with the slider.</li><li><code>interact(..., n_merges=(0, 8, 1))</code> — range and step of the slider.</li></ul>

</details>

## 5. When it breaks: words it has never seen

Swap the character vocabulary for a word-level one and every unseen word becomes a single `<UNK>` symbol. The plot compares what each scheme can encode: characters always cover the word, word tokens keep nothing at all.

In [ ]:
word_vocab = {w: i for i, w in enumerate(words)}   # a word-level vocabulary
print("known words:", list(word_vocab))         # only the three training words
test = ["low", "lower", "lowering", "lowest", "lows"]   # words met at inference time
for w in test:                                  # one line per test word
    print(f"{w:>9} -> {w if w in word_vocab else '<UNK>'}")   # unseen collapses to <UNK>

pos = np.arange(len(test))                      # one x position per test word
char_keep = [1.0 for w in test]                 # every character is in the alphabet
word_keep = [1.0 if w in word_vocab else 0.0 for w in test]   # unseen word -> nothing kept
plt.figure(figsize=(6, 3))                      # wide, short figure
plt.bar(pos - 0.2, char_keep, width=0.4, color="tab:green", label="character tokens")   # always works
plt.bar(pos + 0.2, word_keep, width=0.4, color="tab:red", label="word tokens")          # fails off-vocabulary
plt.xticks(pos, test, rotation=20)              # word names on the x-axis
plt.ylim(0, 1.3)                                # room above the bars
plt.ylabel("fraction of the word encoded")      # y-axis
plt.title("Two of five unseen words become <UNK>")   # the takeaway
plt.legend(loc="upper right")                   # which colour is which
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>word_vocab = {...}</code> — the whole word is now one token.</li><li><code>test = [...]</code> — five words a real user might type.</li><li><code>w if w in word_vocab else '&lt;UNK&gt;'</code> — the only two outcomes at word level.</li><li><code>char_keep</code> — characters always cover the word completely.</li><li><code>word_keep</code> — 1.0 if known, 0.0 if the word is out of vocabulary.</li><li><code>plt.bar(... - 0.2 ...)</code> / <code>plt.bar(... + 0.2 ...)</code> — paired bars per word.</li></ul>

</details>

## Your turn

Change `corpus` in section 1 to `"low lower lowest newer"` and predict which pair wins the first merge before you rerun. Then slide section 4 to the end and check whether `newer` and `lower` ended up sharing a subword.